# Klassification von t0 vs t5

Dies dient als sanity check, ich möchte prüfen, ob meine Pipeline, sowie extrahierten features funktionieren.

--- Bisher nur die catch22 features + hand crafted time domain

In [1]:
from pathlib import Path
import numpy as np
from scripts.myml import loso_binary_nested_cv
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

### Daten Laden

In [2]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

In [3]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2495, 174)
y shape : (2495,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 52 unique subjects
Class counts : {0: 416, 1: 416, 2: 416, 3: 415, 4: 416, 5: 416}


### Hyperparameter Grid

In [4]:
PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42

## Klassifikation t0 vs t5 

### Random Forest vs Random Forest Regressor

In [7]:
# Define models
rf_classifier = RandomForestClassifier(random_state=RANDOM_STATE)
rf_regressor = RandomForestRegressor(random_state=RANDOM_STATE)
models = {
    "Random Forest Classifier": rf_classifier,
    "Random Forest Regressor": rf_regressor
}

# Define comparison
comparison = (0, 5)
model_types = {
    "Random Forest Classifier": "classifier",
    "Random Forest Regressor": "regressor"
}

# Train and evaluate models using LOSO nested CV
for name, model in models.items():

    # Setup for comparison
    mask = (y == comparison[0]) | (y == comparison[1])
    X_filtered = X_catch22[mask]
    y_filtered = y[mask]
    group_labels = subjects[mask]
    y_binary = (y_filtered == comparison[1]).astype(int)


    print(f"\nTraining model: {model}...")
    metrics = loso_binary_nested_cv(
        X_filtered, 
        y_binary, 
        group_labels, 
        model, 
        PARAM_GRID, 
        model_type=model_types[name]
        )
    
    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")
    


Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.895 ± 0.104
    F1       : 0.883 ± 0.134
    AUC      : 0.972 ± 0.053

Training model: RandomForestRegressor(random_state=42)...
    Accuracy : 0.875 ± 0.115
    F1       : 0.870 ± 0.137
    AUC      : 0.955 ± 0.088
